# LLM + NLP Teaching Notebook — No API Key Required

This notebook is designed for students who **cannot use Gemini/OpenAI/Vertex AI APIs** on company laptops.

Everything runs locally with normal Python libraries.

## What students will learn

1. NLP pipeline basics
2. Text cleaning and preprocessing
3. TF-IDF feature creation
4. Logistic Regression sentiment classifier
5. Tokens and vocabulary
6. Embeddings concept using simple vectors
7. Cosine similarity and semantic search
8. Prompt engineering concepts without calling an API
9. RAG architecture using local TF-IDF retrieval
10. PDF/text Q&A style local bot

> This is not using any paid API. It is safe for teaching in restricted environments.


## 0. Install packages

Run this only once. If your company laptop blocks installs, use Google Colab or ask IT to whitelist these packages.


In [ ]:
# Run this cell only if packages are missing
# !pip install scikit-learn pandas numpy matplotlib

## 1. Import libraries

In [ ]:
import re
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.metrics.pairwise import cosine_similarity

print('Libraries imported successfully')

# Lab 8: NLP Pipeline

Topic: **Feature selection, encoding, scaling, TF-IDF, word embeddings, BERT intro, sentiment analysis**

In real NLP projects, raw text is converted into numbers before ML models can understand it.

Basic pipeline:

```text
Raw text → Cleaning → Tokenization → Feature extraction → Model training → Prediction
```


## 2. Create sample dataset

We will use a small sentiment dataset. In real life, this could be customer reviews, support tickets, clinical notes, or call center transcripts.


In [ ]:
data = {
    'text': [
        'The service was excellent and the staff was very helpful',
        'I hated the product, it stopped working after one day',
        'The food was tasty and delivery was quick',
        'Very bad experience, I will never buy again',
        'The movie was amazing with great acting',
        'Worst app ever, too many bugs and crashes',
        'I love this phone, battery life is fantastic',
        'The laptop is slow and the screen quality is poor',
        'Great support team, solved my problem quickly',
        'The hotel room was dirty and noisy',
        'This course is useful and easy to understand',
        'The instructions were confusing and incomplete',
        'Beautiful design and smooth performance',
        'Terrible packaging and delayed delivery',
        'The doctor was kind and explained everything clearly',
        'The medicine caused side effects and discomfort',
        'Fast checkout and excellent customer service',
        'The website keeps failing during payment',
        'I am happy with the quality and price',
        'The product is useless and not worth money'
    ],
    'label': [
        1,0,1,0,1,0,1,0,1,0,
        1,0,1,0,1,0,1,0,1,0
    ]
}

df = pd.DataFrame(data)
df.head()

## 3. Text cleaning

Cleaning removes unnecessary symbols and normalizes text.

Common steps:

- Lowercase text
- Remove special characters
- Remove extra spaces
- Optional: remove stopwords, stemming, lemmatization


In [ ]:
def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['text'].apply(clean_text)
df[['text', 'clean_text', 'label']].head()

## 4. Tokenization

Tokenization means splitting text into smaller units.

Example:

```text
'I love machine learning' → ['i', 'love', 'machine', 'learning']
```

LLMs also tokenize text, but they often use subword tokens instead of full words.


In [ ]:
df['tokens'] = df['clean_text'].apply(lambda x: x.split())
df[['clean_text', 'tokens']].head()

## 5. Bag of Words encoding

Bag of Words creates a vocabulary and counts word frequency.

It ignores word order.


In [ ]:
count_vectorizer = CountVectorizer()
X_count = count_vectorizer.fit_transform(df['clean_text'])

bow_df = pd.DataFrame(X_count.toarray(), columns=count_vectorizer.get_feature_names_out())
bow_df.head()

## 6. TF-IDF

TF-IDF means:

- **TF**: Term Frequency — how often a word appears in a document
- **IDF**: Inverse Document Frequency — how rare or important the word is across documents

Words that appear everywhere get lower weight. Important words get higher weight.


In [ ]:
tfidf = TfidfVectorizer()
X = tfidf.fit_transform(df['clean_text'])

tfidf_df = pd.DataFrame(X.toarray(), columns=tfidf.get_feature_names_out())
tfidf_df.head()

## 7. Train a sentiment classifier

We will train Logistic Regression to classify positive vs negative text.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, df['label'], test_size=0.25, random_state=42, stratify=df['label']
)

model = LogisticRegression()
model.fit(X_train, y_train)

pred = model.predict(X_test)

print('Accuracy:', accuracy_score(y_test, pred))
print('
Classification Report:
')
print(classification_report(y_test, pred, target_names=['Negative', 'Positive']))

## 8. Predict new sentences

In [ ]:
def predict_sentiment(sentence: str):
    cleaned = clean_text(sentence)
    vector = tfidf.transform([cleaned])
    prediction = model.predict(vector)[0]
    probability = model.predict_proba(vector)[0]
    label = 'Positive' if prediction == 1 else 'Negative'
    return label, probability

examples = [
    'This product is amazing and useful',
    'Very poor quality and bad support',
    'The delivery was fast and packaging was good'
]

for e in examples:
    label, prob = predict_sentiment(e)
    print(e)
    print('Prediction:', label, 'Probability:', prob)
    print('-' * 60)

## 9. Word embeddings concept

Traditional ML uses sparse vectors like TF-IDF.

Embeddings are dense vectors that capture meaning.

Example idea:

```text
king    → [0.82, 0.15, 0.44]
queen   → [0.80, 0.18, 0.46]
banana  → [0.10, 0.91, 0.20]
```

Similar words have similar vectors.


In [ ]:
# Tiny handmade embeddings for teaching
embeddings = {
    'king': np.array([0.9, 0.2, 0.1]),
    'queen': np.array([0.88, 0.25, 0.12]),
    'man': np.array([0.7, 0.1, 0.2]),
    'woman': np.array([0.68, 0.15, 0.25]),
    'apple': np.array([0.1, 0.8, 0.3]),
    'banana': np.array([0.12, 0.85, 0.28]),
    'car': np.array([0.2, 0.1, 0.9]),
    'bus': np.array([0.22, 0.12, 0.88])
}

def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

for word in ['queen', 'banana', 'bus']:
    print('similarity king vs', word, '=', round(cosine(embeddings['king'], embeddings[word]), 3))

## 10. BERT intro

BERT is a transformer-based model. It learns contextual embeddings.

Difference:

```text
Static embedding:
bank = same vector always

Contextual embedding:
bank in 'river bank' ≠ bank in 'bank account'
```

BERT is encoder-only and good for classification, search, NER, and embeddings.

GPT-style models are decoder-only and good for generation.


# Lab 9: LLM API Explorer Without API

Topic: **LLM internals, tokens, temperature, context window, embedding geometry, cosine/dot similarity**

Even without API access, students can understand how LLMs work internally.


## 11. Simple tokenizer

This is not a real Gemini/OpenAI tokenizer, but it teaches the idea.


In [ ]:
def simple_tokenize(text):
    return re.findall(r"\w+|[^\w\s]", text.lower())

sample = 'Hello students! Today we are learning LLM tokens.'
tokens = simple_tokenize(sample)
print(tokens)
print('Token count:', len(tokens))

## 12. Context window concept

Context window is the maximum amount of text a model can consider at once.

If token limit is small, old content must be removed or summarized.


In [ ]:
def fit_context(text, max_tokens=10):
    tokens = simple_tokenize(text)
    return tokens[:max_tokens], len(tokens)

long_text = 'LLMs read tokens and predict the next token based on context and probability.'
kept_tokens, total = fit_context(long_text, max_tokens=10)
print('Total tokens:', total)
print('Kept tokens:', kept_tokens)

## 13. Temperature concept using probability sampling

Temperature controls randomness.

- Low temperature: more deterministic
- High temperature: more creative/random


In [ ]:
words = np.array(['cat', 'dog', 'robot', 'teacher', 'cloud'])
logits = np.array([3.0, 2.5, 1.2, 0.8, 0.3])

def softmax_with_temperature(logits, temperature=1.0):
    scaled = logits / temperature
    exp_values = np.exp(scaled - np.max(scaled))
    return exp_values / exp_values.sum()

for temp in [0.2, 0.7, 1.0, 2.0]:
    probs = softmax_with_temperature(logits, temp)
    print('Temperature:', temp)
    for w, p in zip(words, probs):
        print(f'{w}: {p:.3f}')
    print('-' * 40)

## 14. Visualize temperature

In [ ]:
temps = [0.2, 0.7, 1.0, 2.0]
for temp in temps:
    probs = softmax_with_temperature(logits, temp)
    plt.figure(figsize=(6, 4))
    plt.bar(words, probs)
    plt.title(f'Word probability distribution at temperature={temp}')
    plt.xlabel('Candidate next token')
    plt.ylabel('Probability')
    plt.show()

## 15. Dot product vs cosine similarity

Both are used in embedding search.

- Dot product depends on vector direction and magnitude
- Cosine similarity mainly measures direction


In [ ]:
a = np.array([1, 2, 3])
b = np.array([2, 4, 6])
c = np.array([3, 0, 1])

print('Dot a-b:', np.dot(a, b))
print('Cosine a-b:', cosine(a, b))
print('Dot a-c:', np.dot(a, c))
print('Cosine a-c:', cosine(a, c))

# Lab 10: Prompt Library Without API

Topic: **Zero-shot, few-shot, chain-of-thought concept, system prompts, JSON mode, hallucination mitigation**

We will not call an LLM API. Instead, we will create reusable prompt templates that students can later use with any model.


## 16. Prompt template function

In [ ]:
def build_prompt(task, context=None, examples=None, output_format='plain text'):
    prompt = ''
    prompt += 'You are a helpful AI assistant.
'
    prompt += f'Task: {task}
'
    
    if context:
        prompt += f'Context:
{context}
'
    
    if examples:
        prompt += 'Examples:
'
        for i, ex in enumerate(examples, 1):
            prompt += f'Example {i}:
Input: {ex["input"]}
Output: {ex["output"]}
'
    
    prompt += f'Return output as: {output_format}
'
    return prompt

prompt = build_prompt(
    task='Classify the sentiment of the given review.',
    context='Review: The product is useful and easy to use.',
    output_format='JSON with keys: sentiment, reason'
)

print(prompt)

## 17. Zero-shot prompting

Zero-shot means no example is provided.


In [ ]:
zero_shot_prompt = build_prompt(
    task='Translate the sentence into Hindi.',
    context='Sentence: How are you?',
    output_format='Hindi translation only'
)
print(zero_shot_prompt)

## 18. Few-shot prompting

Few-shot means examples are provided.


In [ ]:
examples = [
    {'input': 'I love this app', 'output': '{"sentiment": "positive"}'},
    {'input': 'This is the worst service', 'output': '{"sentiment": "negative"}'},
]

few_shot_prompt = build_prompt(
    task='Classify sentiment.',
    context='Input: Delivery was fast and smooth.',
    examples=examples,
    output_format='JSON only'
)
print(few_shot_prompt)

## 19. Hallucination mitigation prompt

Hallucination means the model invents unsupported information.

Useful instruction:

```text
Use only the provided context. If answer is not available, say "I do not know from the provided context." 
```


In [ ]:
safe_qa_prompt = build_prompt(
    task='Answer the question using only the context.',
    context='Context: Python is a programming language. It is used for data science and web development. Question: Who created Java?',
    output_format='If answer is not in context, say: I do not know from the provided context.'
)
print(safe_qa_prompt)

# Lab 11: PDF Q&A Bot Without API

Topic: **RAG architecture, text chunking, similarity search, top-k retrieval, answer synthesis**

RAG means Retrieval-Augmented Generation.

Production RAG flow:

```text
Documents → Chunking → Embeddings → Vector DB → Retrieve top-k chunks → LLM generates answer
```

No API version:

```text
Documents → Chunking → TF-IDF vectors → Cosine similarity → Return best chunks
```


## 20. Create sample knowledge base

We will simulate PDF content using text paragraphs.


In [ ]:
documents = [
    '''Large Language Models are AI models trained on huge amounts of text. They learn statistical patterns and generate text by predicting the next token.''',
    '''Tokenization is the process of breaking text into smaller units called tokens. Tokens can be words, subwords, or characters.''',
    '''Temperature controls randomness in model output. Lower temperature gives focused answers. Higher temperature gives creative answers.''',
    '''Embeddings are numerical vector representations of text. Similar meanings have similar vectors in embedding space.''',
    '''RAG stands for Retrieval-Augmented Generation. It retrieves relevant information from documents and gives it to the model as context.''',
    '''FAISS and ChromaDB are tools used for vector search. They help find chunks similar to a user query.''',
    '''Prompt engineering is the process of designing instructions that guide an LLM toward better answers.''',
    '''Hallucination happens when a model generates information that is not supported by the provided context.'''
]

kb = pd.DataFrame({'chunk_id': range(len(documents)), 'text': documents})
kb

## 21. Convert chunks into TF-IDF vectors

In [ ]:
rag_vectorizer = TfidfVectorizer(stop_words='english')
rag_matrix = rag_vectorizer.fit_transform(kb['text'])
print('RAG matrix shape:', rag_matrix.shape)

## 22. Retrieve top-k relevant chunks

In [ ]:
def retrieve(query, top_k=3):
    query_vec = rag_vectorizer.transform([query])
    scores = cosine_similarity(query_vec, rag_matrix).flatten()
    top_indices = scores.argsort()[::-1][:top_k]
    results = []
    for idx in top_indices:
        results.append({
            'chunk_id': int(kb.iloc[idx]['chunk_id']),
            'score': float(scores[idx]),
            'text': kb.iloc[idx]['text']
        })
    return results

query = 'What is temperature in LLM?'
retrieve(query, top_k=3)

## 23. Local Q&A bot

Without an LLM, we cannot synthesize human-like answers, but we can return the most relevant context.

This teaches the retrieval part of RAG.


In [ ]:
def local_qa_bot(question, top_k=2):
    results = retrieve(question, top_k=top_k)
    print('Question:', question)
    print('
Most relevant context:')
    for r in results:
        print(f"
Chunk {r['chunk_id']} | Score: {r['score']:.3f}")
        print(r['text'])

local_qa_bot('How does RAG work?', top_k=2)

## 24. Add simple answer generation using extractive method

This is not a real LLM. It simply selects the best sentence from retrieved chunks.


In [ ]:
def extractive_answer(question):
    results = retrieve(question, top_k=3)
    candidate_sentences = []
    for r in results:
        sentences = re.split(r'(?<=[.!?]) +', r['text'])
        candidate_sentences.extend(sentences)
    
    sent_vectorizer = TfidfVectorizer(stop_words='english')
    all_text = candidate_sentences + [question]
    sent_matrix = sent_vectorizer.fit_transform(all_text)
    question_vec = sent_matrix[-1]
    sentence_vecs = sent_matrix[:-1]
    scores = cosine_similarity(question_vec, sentence_vecs).flatten()
    best_idx = scores.argmax()
    return candidate_sentences[best_idx]

questions = [
    'What is tokenization?',
    'What are embeddings?',
    'Why does hallucination happen?',
    'What is vector search?'
]

for q in questions:
    print('Q:', q)
    print('A:', extractive_answer(q))
    print('-' * 70)

# Teaching Notes

## How to explain this to students

### Why no API is used?
Many company laptops block external API access. This notebook teaches the internal logic first.

### What students learn without API

- Text to numbers
- Tokenization
- TF-IDF
- ML classification
- Embeddings concept
- Similarity search
- Prompt design
- RAG retrieval

### What needs API later

- Real answer generation
- Real embeddings
- Function calling
- Multimodal input
- Agent workflows

### Final teaching sequence

```text
NLP basics → TF-IDF → ML model → Tokens → Embeddings → Similarity → Prompting → RAG
```


# Practice Exercises

1. Add 20 more training examples to the sentiment dataset.
2. Try `ngram_range=(1,2)` in `TfidfVectorizer`.
3. Add a new class: Neutral.
4. Create your own prompt templates for SQL generation.
5. Add more chunks to the local RAG knowledge base.
6. Build a simple Streamlit UI on top of `local_qa_bot`.
7. Replace TF-IDF retrieval with sentence-transformer embeddings when internet/package access is available.
